**Navigation** : [Sommaire Sudoku](README.md) | [Version Python](Sudoku-20-CorpusAudit-Python.ipynb) | [Sudoku-07 — Norvig](Sudoku-07-Norvig-Python.ipynb)

# Sudoku-20 : La hiérarchie de difficulté des corpus, mesurée

**Objectifs** : distinguer solution et unicité, comparer propagation et recherche, mesurer la difficulté des corpus et construire un jeu d'indices irrédundant. **Prérequis** : propagation de Norvig (Sudoku-07). **Durée estimée** : 35 minutes.

Ce carnet porte en C# BCL le noyau pédagogique canonique de `scripts/sudoku/core/solvers.py` et les mesures de `corpus.py`. Il lit les trois fichiers réels, sans invoquer Python. Les décisions dépendent de l'ordre d'exploration : le choix MRV parcourt les cases de 0 à 80 et les chiffres de 1 à 9, comme l'organe Python.

## 1. Trois corpus et un noyau de propagation

Les fichiers utilisent `0` ou `.` pour les cases vides. Les unités sont les lignes, colonnes et blocs ; l'élimination propage les candidats uniques et les chiffres qui n'ont qu'une place dans une unité. La lecture vérifie strictement 81 caractères et les symboles admis.

In [1]:
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using System.Text;
using Microsoft.DotNet.Interactive.Formatting;
Console.WriteLine($"Runtime : {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription}");
public static class Sudoku {
    const string Digits = "123456789";
    public static readonly int[][] Units = Enumerable.Range(0,9).Select(r => Enumerable.Range(0,9).Select(c => 9*r+c).ToArray())
        .Concat(Enumerable.Range(0,9).Select(c => Enumerable.Range(0,9).Select(r => 9*r+c).ToArray()))
        .Concat(Enumerable.Range(0,9).Select(b => Enumerable.Range(0,9).Select(k => 9*((b/3)*3+k/3)+(b%3)*3+k%3).ToArray())).ToArray();
    public static readonly int[][][] CellUnits = Enumerable.Range(0,81).Select(i => Units.Where(u => u.Contains(i)).ToArray()).ToArray();
    public static readonly int[][] Peers = Enumerable.Range(0,81).Select(i => CellUnits[i].SelectMany(u => u).Where(j => j != i).Distinct().OrderBy(j => j).ToArray()).ToArray();
    public static int[] Parse(string line) {
        var s = line.Trim();
        if (s.Length != 81 || s.Any(c => c != '.' && (c < '0' || c > '9'))) throw new FormatException("Grille invalide : 81 caractères 0-9 ou point requis");
        return s.Select(c => c == '.' ? 0 : c-'0').ToArray();
    }
    public static bool Valid(int[] p) => p.Length == 81 && p.All(x => x >= 0 && x <= 9) && Units.All(u => { var a = u.Select(i => p[i]).Where(x => x != 0).ToArray(); return a.Distinct().Count() == a.Length; });
    static bool Fill(string[] g, int s, char d) {
        if (g[s] == d.ToString()) return true;
        foreach (char x in g[s].Replace(d.ToString(), "").ToArray()) if (!Eliminate(g,s,x)) return false;
        return true;
    }
    static bool Eliminate(string[] g, int s, char d) {
        if (!g[s].Contains(d)) return true;
        g[s] = g[s].Replace(d.ToString(), "");
        if (g[s].Length == 0) return false;
        if (g[s].Length == 1) foreach (int p in Peers[s]) if (!Eliminate(g,p,g[s][0])) return false;
        foreach (var u in CellUnits[s]) {
            var places = u.Where(i => g[i].Contains(d)).ToArray();
            if (places.Length == 0) return false;
            if (places.Length == 1 && !Fill(g,places[0],d)) return false;
        }
        return true;
    }
    static string[] Propagate(int[] p) {
        if (p.Length != 81 || p.Any(x => x < 0 || x > 9)) throw new ArgumentException("81 chiffres requis");
        var g = Enumerable.Repeat(Digits,81).ToArray();
        for (int i=0;i<81;i++) if (p[i] != 0 && !Fill(g,i,(char)('0'+p[i]))) return null;
        return g;
    }
    static (int count,int decisions,string[] solution) Search(string[] g,int limit) {
        if (g == null) return (0,0,null);
        int best=-1,min=10;
        for (int i=0;i<81;i++) if (g[i].Length > 1 && g[i].Length < min) { min=g[i].Length; best=i; }
        if (best < 0) return (1,0,g);
        int count=0, decisions=0; string[] first=null;
        foreach (char d in g[best]) {
            var copy=(string[])g.Clone();
            var result=Search(Fill(copy,best,d) ? copy : null,limit-count);
            count+=result.count; decisions+=1+result.decisions; first ??= result.solution;
            if (count>=limit) break;
        }
        return (count,decisions,first);
    }
    public static (int count,int decisions,string[] solution) Run(int[] p,int limit) {
        if (limit < 1) throw new ArgumentOutOfRangeException(nameof(limit));
        return Search(Propagate(p),limit);
    }
    public static (int[] reduced,int removed) Reduce(int[] p) {
        var original=Run(p,2); if (original.count != 1) throw new ArgumentException("Une solution unique est requise");
        var g=(int[])p.Clone(); int removed=0;
        for (int i=0;i<81;i++) if (g[i]!=0) {
            int value=g[i]; g[i]=0; var trial=Run(g,2);
            if (trial.count == 1 && trial.solution.SequenceEqual(original.solution)) removed++; else g[i]=value;
        }
        return (g,removed);
    }
}
Console.WriteLine("Noyau de propagation et recherche MRV chargé.");

Runtime : .NET 10.0.14


Noyau de propagation et recherche MRV chargé.


Le compteur explore au plus deux solutions. Une grille contradictoire, une grille vide et un puzzle du corpus servent de contrôles distincts.

In [2]:
var root = new DirectoryInfo(Directory.GetCurrentDirectory());
while (root != null && !Directory.Exists(Path.Combine(root.FullName,"MyIA.AI.Notebooks","Sudoku","Puzzles"))) root=root.Parent;
if (root == null) throw new DirectoryNotFoundException("Corpus Sudoku introuvable depuis le répertoire courant");
var names = new[]{"Sudoku_Easy51.txt","Sudoku_hardest.txt","Sudoku_top95.txt"};
var corpora = names.ToDictionary(n => n,n => File.ReadAllLines(Path.Combine(root.FullName,"MyIA.AI.Notebooks","Sudoku","Puzzles",n)).Where(s => !string.IsNullOrWhiteSpace(s)).Select(Sudoku.Parse).ToArray());
foreach (var n in names) {
 var firstLine = File.ReadLines(Path.Combine(root.FullName,"MyIA.AI.Notebooks","Sudoku","Puzzles",n)).First(s => !string.IsNullOrWhiteSpace(s)).Trim();
 Console.WriteLine($"{n,-22} {corpora[n].Length,3} grilles | première ligne : {firstLine.Substring(0,24)}...");
}
var easy = corpora[names[0]][0];
var invalid=(int[])easy.Clone(); invalid[0]=1; invalid[1]=1;
Console.WriteLine($"Contradictoire : {Sudoku.Run(invalid,2).count} ; vide : {Sudoku.Run(new int[81],2).count} ; premier Easy51 : {Sudoku.Run(easy,2).count}");

Sudoku_Easy51.txt       51 grilles | première ligne : 902005403100063025508407...


Sudoku_hardest.txt      11 grilles | première ligne : 85...24..72......9..4......


Sudoku_top95.txt        95 grilles | première ligne : 4.....8.5.3..........7.....


Contradictoire : 0 ; vide : 2 ; premier Easy51 : 1


La grille vide atteint le plafond 2 : elle possède au moins deux solutions. Une seule solution pour le premier puzzle signifie que la recherche complète n'en a pas trouvé de seconde. La contradiction détectée ne donne aucune solution.

### Exercice 1 — notation des cases vides

Écrire une fonction qui distingue `0` et `.` sur des lignes représentatives, puis vérifier chaque ligne de chaque fichier. La réponse doit être l'une de ces deux notations ; tester aussi les deux cas.

In [3]:
string NotationVide(string[] lignes) { return null; } // TODO étudiant
void VerifierNotation(Func<string[],string> f) {
    var essais = new List<(string[] lines,string expected)> {
        (new[]{"0" + new string('1',80)}, "0"),
        (new[]{"." + new string('1',80)}, ".")
    };
    foreach (var n in names) {
        var lignes = File.ReadAllLines(Path.Combine(root.FullName,"MyIA.AI.Notebooks","Sudoku","Puzzles",n)).Where(s=>!string.IsNullOrWhiteSpace(s)).ToArray();
        string expected = n == names[0] ? "0" : ".";
        essais.Add((lignes,expected));
        foreach (var ligne in lignes) essais.Add((new[]{ligne},expected));
    }
    var answers = essais.Select(t => f((string[])t.lines.Clone())).ToArray();
    if (answers.All(x=>x == null)) { Console.WriteLine("Exercice 1 à compléter"); return; }
    for (int i=0;i<essais.Count;i++)
        if (answers[i] != essais[i].expected) throw new InvalidOperationException($"Exercice 1 : notation incorrecte pour le test {i+1}");
    Console.WriteLine($"Exercice 1 : {essais.Count} tests réussis, dont les 157 lignes réelles");
}
VerifierNotation(NotationVide);

Exercice 1 à compléter


## 2. Audit des 157 grilles

Pour chaque grille, le comptage est plafonné à deux, tandis que l'effort mesure les branches essayées jusqu'à la première solution. L'ordre des chiffres et les égalités MRV influencent ces compteurs.

In [4]:
var chrono = System.Diagnostics.Stopwatch.StartNew();
var records = names.ToDictionary(n => n, n => corpora[n].Select((p,i) => {
    var count=Sudoku.Run(p,2).count; var first=Sudoku.Run(p,1);
    return (index:i+1, clues:p.Count(x=>x!=0), valid:Sudoku.Valid(p), solutions:count, decisions:first.count==0 ? -1 : first.decisions);
}).ToArray());
chrono.Stop();
Console.WriteLine($"Audit des trois corpus : {chrono.Elapsed.TotalSeconds:F3} s (durée dépendante de la machine)");
foreach (var n in names) {
 var rs=records[n]; var ds=rs.Select(r=>r.decisions).OrderBy(x=>x).ToArray(); var cs=rs.Select(r=>r.clues).OrderBy(x=>x).ToArray();
 Console.WriteLine($"{n,-22} n={rs.Length} invalides={rs.Count(r=>!r.valid)} non_uniques={rs.Count(r=>r.solutions!=1)} propagation={ds.Count(x=>x==0)} médiane={ds[ds.Length/2]} max={ds.Last()} indices={cs.First()}..{cs.Last()}");
}

Audit des trois corpus : 2.238 s (durée dépendante de la machine)


Sudoku_Easy51.txt      n=51 invalides=0 non_uniques=0 propagation=41 médiane=0 max=7 indices=22..45


Sudoku_hardest.txt     n=11 invalides=0 non_uniques=0 propagation=1 médiane=8 max=21 indices=22..28


Sudoku_top95.txt       n=95 invalides=0 non_uniques=0 propagation=0 médiane=39 max=446 indices=17..26


La propagation seule résout **41/51** grilles d'Easy51 (environ 80,4 %), **1/11** de hardest (environ 9,1 %) et **0/95** de top95. Les médianes d'effort sont respectivement **0, 8 et 39** décisions ; les maxima sont **7, 21 et 446**. Les plages d'indices **22–45, 22–28 et 17–26** se recouvrent : le nombre d'indices ne suffit pas à prédire l'effort. La durée affichée mesure l'audit sur cette machine, sans valeur universelle.

### Exercice 2 — ordre de la hiérarchie

Écrire un prédicat qui vérifie les parts de propagation décroissantes et les maxima croissants, égalités admises. Des tableaux de longueurs différentes ne décrivent pas une même série et doivent rendre `false`. Les tests couvrent des ordres vrais et faux, les égalités et cette incompatibilité.

In [5]:
bool? Hierarchie(double[] parts, int[] maxima) { return null; } // TODO étudiant
void VerifierHierarchie(Func<double[],int[],bool?> f) {
    var cas = new (double[] p,int[] m,bool expected)[] {
        (new[]{0.8,0.1,0.0},new[]{7,21,446},true),
        (new[]{0.2,0.6,0.0},new[]{7,21,446},false),
        (new[]{0.8,0.1,0.0},new[]{7,3,446},false),
        (new[]{0.8,0.8,0.0},new[]{7,7,446},true),
        (new[]{0.8,0.1},new[]{7,21,446},false),
        (new[]{0.0,0.1,0.8},new[]{446,21,7},false),
        (new[]{0.9,0.3,0.1},new[]{2,5,9},true)
    };
    var reponses = cas.Select(t=>f((double[])t.p.Clone(),(int[])t.m.Clone())).ToArray();
    if (reponses.All(x=>x == null)) { Console.WriteLine("Exercice 2 à compléter"); return; }
    for (int i=0;i<cas.Length;i++) if (reponses[i] != cas[i].expected)
        throw new InvalidOperationException($"Exercice 2 : résultat incorrect au test {i+1}");
    Console.WriteLine($"Exercice 2 : {cas.Length} tests réussis");
}
var parts = names.Select(n => records[n].Count(r => r.decisions==0)/(double)records[n].Length).ToArray();
var maxima = names.Select(n => records[n].Max(r=>r.decisions)).ToArray();
VerifierHierarchie(Hierarchie);

Exercice 2 à compléter


## 3. Contrôle négatif

Le nom `hardest` ne garantit pas que chaque grille exige une recherche. Cherchons les indices des grilles résolues par propagation seule.

In [6]:
foreach (var n in names.Skip(1)) {
 var hits=records[n].Where(r=>r.decisions==0).ToArray();
 Console.WriteLine($"{n} : {(hits.Length==0 ? "aucune" : string.Join(", ",hits.Select(r=>$"n° {r.index} ({r.clues} indices)")))}");
}

Sudoku_hardest.txt : n° 5 (22 indices)


Sudoku_top95.txt : aucune


Le contrôle trouve la grille **n° 5 de hardest**, avec **22 indices**, résolue par propagation seule ; aucune grille de `top95` ne l'est. Cette propriété précise ne suffit pas à qualifier toute la difficulté d'un corpus ou de chacune de ses grilles.

## 4. Réduire les indices

On essaie les suppressions dans l'ordre des cases, en conservant seulement celles qui maintiennent une solution unique identique. Une fois la réduction achevée, retirer un indice supplémentaire détruit l'unicité : c'est l'irrédundance, non la preuve d'un minimum de cardinal.

In [7]:
var reduced = new Dictionary<int,int[]>();
foreach (int number in new[]{1,8}) {
 var original=corpora[names[0]][number-1]; var result=Sudoku.Reduce(original); reduced[number]=result.reduced;
 var before=Sudoku.Run(original,2); var after=Sudoku.Run(result.reduced,2);
 bool irredundant=Enumerable.Range(0,81).Where(i=>result.reduced[i]!=0).All(i=>{var t=(int[])result.reduced.Clone();t[i]=0;return Sudoku.Run(t,2).count!=1;});
 Console.WriteLine($"Easy51 n° {number} : {original.Count(x=>x!=0)} -> {result.reduced.Count(x=>x!=0)} indices ; retirés={result.removed} ; unique={after.count==1} ; même solution={after.solution.SequenceEqual(before.solution)} ; irrédundant={irredundant}");
}

Easy51 n° 1 : 45 -> 24 indices ; retirés=21 ; unique=True ; même solution=True ; irrédundant=True


Easy51 n° 8 : 26 -> 24 indices ; retirés=2 ; unique=True ; même solution=True ; irrédundant=True


Easy51 n° 1 passe de **45 à 24 indices**, et n° 8 de **26 à 24**. Les deux réductions conservent la solution unique ; le test de chaque indice restant établit leur irrédundance. L'ordre des suppressions peut produire un autre ensemble ; aucun minimum global n'est démontré.

### Exercice 3 — retirer un indice

Rendre le nombre de solutions plafonné à deux après chaque retrait possible. Les clés sont les indices présents et la grille reçue doit rester inchangée. La vérification porte sur une grille réduite, la grille d'origine et une grille contradictoire, avec des résultats allant de 0 à 2.

In [8]:
Dictionary<int,int> Retraits(int[] puzzle) { return null; } // TODO étudiant
void VerifierRetraits(Func<int[],Dictionary<int,int>> f) {
    var cas = new[]{(int[])reduced[1].Clone(), (int[])easy.Clone(), (int[])invalid.Clone()};
    var reponses = new List<Dictionary<int,int>>();
    foreach (var original in cas) {
        var argument=(int[])original.Clone(); var reponse=f(argument);
        if (!argument.SequenceEqual(original)) throw new InvalidOperationException("Exercice 3 : la grille fournie a été modifiée");
        reponses.Add(reponse);
    }
    if (reponses.All(x=>x == null)) { Console.WriteLine("Exercice 3 à compléter"); return; }
    for (int k=0;k<cas.Length;k++) {
        var puzzle=cas[k]; var donne=reponses[k];
        var indices=Enumerable.Range(0,81).Where(i=>puzzle[i]!=0).ToArray();
        if (donne == null || !donne.Keys.OrderBy(i=>i).SequenceEqual(indices))
            throw new InvalidOperationException($"Exercice 3 : clés incorrectes pour le cas {k+1}");
        foreach (int i in indices) {
            var candidate=(int[])puzzle.Clone(); candidate[i]=0;
            if (donne[i] != Sudoku.Run(candidate,2).count)
                throw new InvalidOperationException($"Exercice 3 : comptage incorrect au cas {k+1}, case {i}");
        }
    }
    Console.WriteLine("Exercice 3 : grilles réduite, originale et contradictoire vérifiées");
}
VerifierRetraits(Retraits);

Exercice 3 à compléter


## 5. Deux vues de la hiérarchie

La part de grilles résolues par propagation est en pourcentage ; l'effort maximal utilise une échelle logarithmique en décisions. Les deux indicateurs évoluent en sens opposés selon les corpus.

In [9]:
public record SvgFigure(string Content);
Formatter.Register<SvgFigure>((v,w)=>w.Write(v.Content),"text/html");
string F(double x)=>x.ToString("0.##",CultureInfo.InvariantCulture);
var svg=new StringBuilder(); svg.Append("<svg xmlns='http://www.w3.org/2000/svg' width='920' height='400' viewBox='0 0 920 400'><rect width='920' height='400' fill='white'/>");
void Text(double x,double y,string s,int size=13) => svg.Append($"<text x='{F(x)}' y='{F(y)}' font-family='sans-serif' font-size='{size}' text-anchor='middle'>{System.Security.SecurityElement.Escape(s)}</text>");
void Line(double x1,double y1,double x2,double y2) => svg.Append($"<line x1='{F(x1)}' y1='{F(y1)}' x2='{F(x2)}' y2='{F(y2)}' stroke='black'/>");
Text(230,27,"Propagation seule (%)",16); Text(690,27,"Pire effort (décisions, log10)",16);
for(int panel=0;panel<2;panel++) { int left=70+panel*460; Line(left,55,left,330); Line(left,330,left+340,330);
 for(int j=0;j<=4;j++){ double val=panel==0 ? j*25 : Math.Pow(10,j); double y=330-j*65; Line(left-5,y,left+340,y); Text(left-24,y+4,panel==0?F(val):F(val),11); }
 for(int i=0;i<3;i++){double value=panel==0?parts[i]*100:maxima[i]; double height=panel==0?value/100*260:Math.Log10(Math.Max(value,1))/4*260; double x=left+45+i*105;
 svg.Append($"<rect x='{F(x)}' y='{F(330-height)}' width='55' height='{F(height)}' fill='{(panel==0?"#4477aa":"#cc5566")}'/>"); Text(x+27,Math.Max(50,322-height),panel==0?F(value)+" %":F(value)); Text(x+27,350,new[]{"Easy51","hardest","top95"}[i]); }
}
svg.Append("</svg>"); display(new SvgFigure(svg.ToString()));

Propagation seule (%) Pire effort (décisions, log10) 0 25 50 75 100 80.39 % Easy51 9.09 % hardest 0 % top95 1 10 100 1000 10000 7 Easy51 21 hardest 446 top95

La part de propagation diminue tandis que le pire effort augmente pour ces corpus et ce solveur. Le panneau logarithmique montre les écarts sans attribuer les décisions à tous les puzzles du corpus.

## À retenir

- Un résultat plafonné à 2 distingue l'unicité de plusieurs solutions.
- Les parts et les maxima caractérisent des échantillons ; les compteurs dépendent de la stratégie.
- L'irrédundance ne prouve pas le plus petit nombre possible d'indices.

## Vérifiez votre compréhension

1. Pourquoi une première solution ne prouve-t-elle pas l'unicité ?
2. Que signifie une grille de `hardest` résolue par propagation seule ?
3. Pourquoi une réduction irrédundante n'est-elle pas nécessairement minimale ?

<details><summary>Éléments de réponse</summary>

1. Une autre solution peut exister. 2. Cette grille ne nécessite aucune décision pour ce solveur, sans caractériser toute sa difficulté. 3. Un autre ensemble d'indices pourrait être plus petit.
</details>

## Pour aller plus loin

[Sudoku-07 — Propagation de Norvig](Sudoku-07-Norvig-Python.ipynb) et [Sudoku-18 — Comparaison des solveurs](Sudoku-18-Comparison-Python.ipynb).